> **Disclaimer:** This workshop is provided for educational and informational purposes only. The sample code and configurations are not intended for production use. Please review and adapt them according to your organization's security and compliance requirements. AWS service charges may apply for resources created during this workshop. Video content used in this workshop is sourced from publicly available materials and is attributed where applicable.

# TwelveLabs Pegasus on Amazon Bedrock Workshop

TwelveLabs is a leading provider of multimodal AI models specializing in video understanding and analysis. Amazon Bedrock now offers TwelveLabs Pegasus, which provides comprehensive video understanding and analysis. This model empowers developers to build applications that can intelligently process, analyze, and derive insights from video data at scale.

TwelveLabs models analyze and combine information from all modalities (visual, audio, text) to accurately interpret videos holistically — similar to how humans watch, listen, and read simultaneously.

Supported modalities:
- **Visual**: Actions, objects, events, OCR, brand logos
- **Audio**: Ambient sounds, music, human speech

## Part 0: Setup

### Dependencies

In [ ]:
%pip install -r requirements.txt -Uq

In [ ]:
import boto3, botocore
import json
import re
import pandas as pd
import numpy as np
import uuid
import time
import base64
from IPython.display import clear_output, HTML, display, Image
from sklearn.metrics.pairwise import cosine_similarity
from opensearchpy import AWSV4SignerAuth, OpenSearch, RequestsHttpConnection

### Configure boto3

In [ ]:
# Initialize AWS session
session = boto3.Session() # TODO: (Optional) Replace with your AWS profile, keep as is to use the default profile

In [ ]:
# Get AWS Region from session
AWS_REGION = session.region_name
# AWS_REGION = "us-east-1" # OPTIONAL: Manual override for workshop region

print(f"AWS Region: {AWS_REGION}")

workshop_supported_regions = [
    "us-east-1", # N. Virginia
    "eu-west-1", # Ireland
    "ap-northeast-2" # Seoul
]

if AWS_REGION not in workshop_supported_regions:
    raise ValueError(f"AWS Region {AWS_REGION} is not supported for this workshop. Please use one of the following regions: {workshop_supported_regions}")

In [ ]:
# Initialize required AWS clients
bedrock_client = session.client('bedrock-runtime')
s3_client = session.client('s3')

### Configure S3 bucket

In [ ]:
# S3 Configuration
S3_BUCKET_NAME = "<YOUR_S3_BUCKET_NAME>" # TODO: Replace with your S3 bucket name
S3_VIDEOS_PATH = "videos"
S3_IMAGES_PATH = "images"
S3_EMBEDDINGS_PATH = "embeddings"

# Validate S3 bucket name
if S3_BUCKET_NAME == "<YOUR_S3_BUCKET_NAME>" or S3_BUCKET_NAME == "":
    raise ValueError("Please replace <YOUR_S3_BUCKET_NAME> with your S3 bucket name")

---
## Part 1: Setting up Video Data in S3


### Part 2a: Storing videos in S3

#### Set up sample dataset to S3 bucket

In [ ]:
# AWS Account ID for S3 bucket ownership
aws_account_id = session.client('sts').get_caller_identity()["Account"]

print(f"AWS Account ID: {aws_account_id}")
print(f"S3 Bucket: {S3_BUCKET_NAME}")
print(f"S3 Videos Path: {S3_VIDEOS_PATH}")
print(f"S3 Images Path: {S3_IMAGES_PATH}")
print(f"S3 Embeddings Path: {S3_EMBEDDINGS_PATH}")

# Verify bucket access
try:
    s3_client.head_bucket(Bucket=S3_BUCKET_NAME)
    print(f"✅ Successfully connected to S3 bucket: {S3_BUCKET_NAME}")
except Exception as e:
    print(f"❌ Error accessing S3 bucket: {e}")
    print("Please ensure the bucket exists and you have proper permissions.")

#### Netflix Open Content

The [Netflix Open Content](https://opencontent.netflix.com/) is an open source content available under the [Creative Commons Attribution 4.0 International Public License](https://www.google.com/url?q=https%3A%2F%2Fcreativecommons.org%2Flicenses%2Fby%2F4.0%2Flegalcode&sa=D&sntz=1&usg=AOvVaw3DDX6ldzWtAO5wOs5KkByf).

The assets are available for download at: http://download.opencontent.netflix.com/

We will be utilizing a subset of the videos for demonstrating how to utilize the TwelveLabs models on Amazon Bedrock.

In [ ]:
# Sample video S3 URIs
sample_videos = [
    # 's3://download.opencontent.netflix.com/TechblogAssets/CosmosLaundromat/encodes/CosmosLaundromat_2048x858_24fps_SDR.mp4',
    # 's3://download.opencontent.netflix.com/TechblogAssets/Meridian/encodes/Meridian_3840x2160_5994fps_SDR.mp4',
    's3://download.opencontent.netflix.com/TechblogAssets/Sparks/encodes/Sparks_4096x2160_5994fps_SDR.mp4'
]

In [ ]:
# Unsigned S3 client
public_s3_client = boto3.client('s3', config=botocore.client.Config(signature_version=botocore.UNSIGNED))

In [ ]:
def parse_s3_uri(s3_uri: str) -> tuple[str, str]:
    """
    Parses an S3 URI like s3://bucket-name/path/to/object and returns (bucket, key)

    Args:
        s3_uri (str): The S3 URI to parse
        
    Returns:
        tuple[str, str]: The bucket and key
    """
    pattern = r'^s3://([^/]+)/(.+)$'
    match = re.match(pattern, s3_uri)
    if not match:
        raise ValueError(f"Invalid S3 URI format: {s3_uri}")
    return match.group(1), match.group(2)


def copy_public_s3_object_to_private_bucket(public_s3_uri: str, dest_bucket: str, dest_key: str, aws_profile: str = 'default') -> None:
    """
    Copies a public S3 object to a private bucket

    Args:
        public_s3_uri (str): The S3 URI of the public object to copy
        dest_bucket (str): The name of the private bucket to copy to
        dest_key (str): The key of the object to copy to
        aws_profile (str): The AWS profile to use for the authenticated client
    """

    # Parse source bucket and key
    source_bucket, source_key = parse_s3_uri(public_s3_uri)

    # Anonymous client to read public object
    anon_s3 = boto3.client('s3', config=botocore.client.Config(signature_version=botocore.UNSIGNED))

    print(f"Downloading from {public_s3_uri}...")
    response = anon_s3.get_object(Bucket=source_bucket, Key=source_key)
    data = response['Body'].read()

    print(f"Uploading to s3://{dest_bucket}/{dest_key} ...")
    s3_client.put_object(Bucket=dest_bucket, Key=dest_key, Body=data)

    print("✅ Copy completed successfully!")

In [ ]:
# Copy videos to the S3 bucket
for video_uri in sample_videos:
    # Extract the filename from the S3 key
    _, src_key = parse_s3_uri(video_uri)
    filename = src_key.split("/")[-1]
    dest_key = f"{S3_VIDEOS_PATH}/{filename}"
    copy_public_s3_object_to_private_bucket(
        public_s3_uri=video_uri,
        dest_bucket=S3_BUCKET_NAME,
        dest_key=dest_key
    )

## Part 4: Using Pegasus on Bedrock

### Select the video to analyze

In [ ]:
s3_response = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME, Prefix=S3_VIDEOS_PATH)

# List all object keys
if 'Contents' in s3_response:
    object_keys = [obj['Key'] for obj in s3_response['Contents']]
    for key in object_keys:
        print(key)
    print(f"\nTotal objects found: {len(object_keys)}")
else:
    print("No objects found in the specified bucket and prefix.")

In [ ]:
video_s3_key = "videos/Sparks_4096x2160_5994fps_SDR.mp4" # TODO: (Optional) Replace with your video S3 key

# Validate video S3 key
if video_s3_key == "<YOUR_VIDEO_S3_KEY>" or video_s3_key == "":
    raise ValueError("Please replace <YOUR_VIDEO_S3_KEY> with your video S3 key")

### View the video

In [ ]:
# Generate presigned URL for the video
presigned_url = s3_client.generate_presigned_url(
    "get_object",
    Params={"Bucket": S3_BUCKET_NAME, "Key": video_s3_key},
    ExpiresIn=3600
)

# Play the video
play_video(presigned_url, 0)

### Part 4a: Analyze with Pegasus on Bedrock

#### TwelveLabs Pegasus

Pegasus is a generative model for video-to-text generation. It analyzes visual, audio, and textual modalities to generate contextually relevant text — including summaries, highlights, timestamps, and custom analyses.

#### Pegasus 1.2 on Bedrock

- Model ID: `twelvelabs.pegasus-1-2-v1:0`
- Input: Video
- Output: Text
- Max video: 1 hour (< 2GB)
- Supports `InvokeModel` and `InvokeModelWithResponseStream` (streaming)

**Resources:** [AWS Docs](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-pegasus.html) | [TwelveLabs Docs](https://docs.twelvelabs.io/v1.3/docs/concepts/models/pegasus)

In [ ]:
PEGASUS_MODEL_ID_REGIONS = {
    "us-east-1": "us.twelvelabs.pegasus-1-2-v1:0",
    "us-west-2": "us.twelvelabs.pegasus-1-2-v1:0",
    "eu-west-1": "eu.twelvelabs.pegasus-1-2-v1:0",
    "ap-northeast-2": "apac.twelvelabs.pegasus-1-2-v1:0"
}

In [ ]:
try:
    PEGASUS_MODEL_ID = PEGASUS_MODEL_ID_REGIONS[AWS_REGION]
    print(PEGASUS_MODEL_ID)
except KeyError:
    raise ValueError(f"Pegasus 1.2 is not supported for {AWS_REGION}")

##### Analyze a video with the InvokeModel API

In [ ]:
prompt = "What is the video about?"

request_body = {
    "inputPrompt": prompt,
    "mediaSource": {
        "s3Location": {
            "uri": f"s3://{S3_BUCKET_NAME}/{video_s3_key}",
            "bucketOwner": aws_account_id
        }
    },
    "temperature": 0
}

response = bedrock_client.invoke_model(
    modelId=PEGASUS_MODEL_ID,
    body=json.dumps(request_body),
    contentType="application/json",
    accept="application/json"
)

# Parse the response and print the model outputs
response_body = json.loads(response.get("body").read())
print(response_body["message"])

### Part 4b: Pegasus streaming response

In [ ]:
prompt = "What is the video about?"

request_body = {
    "inputPrompt": prompt,
    "mediaSource": {
        "s3Location": {
            "uri": f"s3://{S3_BUCKET_NAME}/{video_s3_key}",
            "bucketOwner": aws_account_id
        }
    },
    "temperature": 0
}

streaming_response = bedrock_client.invoke_model_with_response_stream(
    modelId=PEGASUS_MODEL_ID,
    body=json.dumps(request_body),
    contentType="application/json",
    accept="application/json"
)

# Extract and print the response text in real-time.
message = ""
for event in streaming_response["body"]:
    chunk = json.loads(event["chunk"]["bytes"])
    print(chunk["message"], end="")

print(message)

## Part 5: Video Analysis with Pegasus

Pegasus can generate various types of text output from videos:
- **Summaries** — Key points of a video
- **Topics and hashtags** — Quick content breakdown
- **Highlights** — Key events in order
- **Chapters** — Topic-based chronological breakdown
- **Open-ended text** — Custom outputs based on your prompts (reports, action items, marketing copy, etc.)

### Part 5a: Summaries, hashtags, and highlights

In [ ]:
# Generate a summary of the video
prompt = "Summarize the video"

request_body = {
    "inputPrompt": prompt,
    "mediaSource": {
        "s3Location": {
            "uri": f"s3://{S3_BUCKET_NAME}/{video_s3_key}",
            "bucketOwner": aws_account_id
        }
    },
    "temperature": 0
}

streaming_response = bedrock_client.invoke_model_with_response_stream(
    modelId=PEGASUS_MODEL_ID,
    body=json.dumps(request_body),
    contentType="application/json",
    accept="application/json"
)

# Extract and print the response text in real-time.
message = ""
for event in streaming_response["body"]:
    chunk = json.loads(event["chunk"]["bytes"])
    print(chunk["message"], end="")

print(message)

In [ ]:
# Generate relevant hashtags for the video
prompt = "Generate hashtags for the video"

request_body = {
    "inputPrompt": prompt,
    "mediaSource": {
        "s3Location": {
            "uri": f"s3://{S3_BUCKET_NAME}/{video_s3_key}",
            "bucketOwner": aws_account_id
        }
    },
    "temperature": 0
}

streaming_response = bedrock_client.invoke_model_with_response_stream(
    modelId=PEGASUS_MODEL_ID,
    body=json.dumps(request_body),
    contentType="application/json",
    accept="application/json"
)

# Extract and print the response text in real-time.
message = ""
for event in streaming_response["body"]:
    chunk = json.loads(event["chunk"]["bytes"])
    print(chunk["message"], end="")

print(message)

In [ ]:
# Generate highlights of the video
prompt = "What are the highlighted moments of this video?"

request_body = {
    "inputPrompt": prompt,
    "mediaSource": {
        "s3Location": {
            "uri": f"s3://{S3_BUCKET_NAME}/{video_s3_key}",
            "bucketOwner": aws_account_id
        }
    },
    "temperature": 0
}

streaming_response = bedrock_client.invoke_model_with_response_stream(
    modelId=PEGASUS_MODEL_ID,
    body=json.dumps(request_body),
    contentType="application/json",
    accept="application/json"
)

# Extract and print the response text in real-time.
message = ""
for event in streaming_response["body"]:
    chunk = json.loads(event["chunk"]["bytes"])
    print(chunk["message"], end="")

print(message)

### Part 5b: Structured outputs

Structured outputs for Pegasus lets users specify the structured output format as a JSON schema. Structured outputs can be useful for building automated integrations to application workflows such as metadata enrichment.

In [ ]:
# Using JSON Schema to generate structured output
prompt = """
Generate metadata for the video with the following fields:
- title: (string) The title of the video
- description: (string) The description of the video
- mood: (string) The mood of the video
- genre: (string) The genre of the video
"""

request_body = {
    "inputPrompt": prompt,
    "mediaSource": {
        "s3Location": {
            "uri": f"s3://{S3_BUCKET_NAME}/{video_s3_key}",
            "bucketOwner": aws_account_id
        }
    },
    "temperature": 0,
    "responseFormat": {
        "jsonSchema": {
            "type": "object",
            "properties": {
                "title": {
                    "type": "string"
                },
                "description": {
                    "type": "string"
                },
                "mood": {
                    "type": "string"
                },
                "genre": {
                    "type": "string"
                }
            },
            "required": ["title", "description", "mood", "genre"]
        }
    }
}

response = bedrock_client.invoke_model(
    modelId=PEGASUS_MODEL_ID,
    body=json.dumps(request_body),
    contentType="application/json",
    accept="application/json"
)

# Parse the response and print the model outputs
response_body = json.loads(response.get("body").read())
message_data = json.loads(response_body["message"])
print(json.dumps(message_data, indent=4))

---
## Cleanup


#### Empty S3 bucket

In [ ]:
# List objects and prepare for deletion
response = s3_client.list_objects_v2(Bucket=S3_BUCKET_NAME)

# Empty S3 bucket
try:
    if 'Contents' in response:
        objects_to_delete = [{'Key': obj['Key']} for obj in response['Contents']]
        s3_client.delete_objects(
            Bucket=S3_BUCKET_NAME,
            Delete={'Objects': objects_to_delete}
        )
        print(f"✅ Bucket '{S3_BUCKET_NAME}' emptied successfully.")
    else:
        print(f"✅ Bucket '{S3_BUCKET_NAME}' is already empty.")
except Exception as e:
    print(f"❌ Error emptying bucket '{S3_BUCKET_NAME}': {e}")